# FPL Points Prediction — Colab Runner

The retraining does not fit on an 8 GB laptop: the feature-engineering step expands
~230,000 rows to ~230 columns, and the models need `xgboost` and `lightgbm`. This
notebook runs the whole fixed pipeline on Colab instead.

**Runtime → Change runtime type → High-RAM** if your account offers it. A GPU is not
useful here — scikit-learn, XGBoost and LightGBM all run on CPU for this workload, so
pick a **CPU** runtime with as much RAM as you can get.

| stage | script | roughly |
| --- | --- | --- |
| 1. repair 2024-25 merged_gw | `rebuild_merged_gw.py` | seconds |
| 2. rebuild the dataset | `build_dataset.py` | 2–5 min |
| 3. feature engineering | `build_features.py` | 10–25 min, RAM-hungry |
| 4. train + save models | `train.py` | 30–90 min |

Each script is safe to re-run, and each stage writes its output to disk, so you can
stop after any stage and pick up later.

## 0. Runtime check

In [ ]:
import multiprocessing, os, shutil, sys, platform

try:
    import psutil
    ram = psutil.virtual_memory().total / 1e9
except ImportError:
    ram = float(os.popen("awk '/MemTotal/ {print $2}' /proc/meminfo").read() or 0) / 1e6

print(f"python {sys.version.split()[0]} on {platform.platform()}")
print(f"CPUs:  {multiprocessing.cpu_count()}")
print(f"RAM:   {ram:.1f} GB")
print(f"disk:  {shutil.disk_usage('/').free / 1e9:.0f} GB free")

if ram < 11:
    print("\nWARNING: under ~11 GB. Stage 3 may be killed.")
    print("Runtime -> Change runtime type -> High-RAM, or run stage 3 with fewer seasons.")

## 1. Get the project into Colab

Two ways. **Drive** is the one to use if you will run this more than once — the outputs
persist between sessions, and the data is ~500 MB to re-upload otherwise.

Put the project folder in your Drive (e.g. `MyDrive/fpl/`) so that `final.ipynb` and
`data/` sit directly inside it, then run the Drive cell.

In [ ]:
# --- Option A: Google Drive (recommended) ---
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/fpl'   # <-- edit to match your folder
%cd $PROJECT
!ls

In [ ]:
# --- Option B: upload a zip of the project instead ---
# Skip this cell if you used Drive above.
#
# from google.colab import files
# up = files.upload()                     # pick your project .zip
# !unzip -q -o "$(ls *.zip | head -1)" -d /content/fpl
# PROJECT = '/content/fpl'
# %cd $PROJECT
# !ls

In [ ]:
import os
for required in ('final.ipynb', 'scripts', 'data'):
    assert os.path.exists(required), (
        f"{required!r} not found in {os.getcwd()}. "
        f"Point PROJECT at the folder holding final.ipynb."
    )
print(f"project root: {os.getcwd()}")
print(f"seasons: {sorted(d for d in os.listdir('data') if d[0].isdigit())}")

## 2. Dependencies

In [ ]:
!pip install -q xgboost lightgbm pulp
import xgboost, lightgbm, sklearn, pandas, numpy
for m in (pandas, numpy, sklearn, xgboost, lightgbm):
    print(f"{m.__name__:<12} {m.__version__}")

## 3. Repair `data/2024-25/gws/merged_gw.csv`

That file was concatenated without aligning columns by name. FPL added seven `mng_*`
columns in GW22 of 2024-25, so every row from GW22 on carried 49 fields against a
42-field header — and every reader in this project opens it with
`on_bad_lines='skip'`, silently discarding 13,427 rows (49% of the season).

Run with no arguments first to audit every season, then repair.

In [ ]:
!python scripts/rebuild_merged_gw.py

In [ ]:
!python scripts/rebuild_merged_gw.py --season 2024-25 --write

## 4. Rebuild the dataset

Re-merges every season (final.ipynb cells 3–34) so the recovered rows actually reach
the training data, then carries the FBref defensive columns over from the previous
`all_seasons_data_final.csv` by joining on `(season, element, fixture)`.

The 13,105 recovered rows never had FBref stats merged, so they get zeros and
`has_fbref_defensive=0` — visible rather than silently mixed in.

In [ ]:
!python scripts/build_dataset.py --write

## 5. Feature engineering

The RAM-hungry stage: lagged previous-game stats, opponent strength, rolling player
form and context features, ~230 columns out.

If the runtime dies here, restart and re-run from this cell — stage 4's output is
already on disk.

In [ ]:
!python scripts/build_features.py

## 6. Train

Runs the notebook's own fixed training cells, so the fixes apply by construction:

- direct models train on the **featured** frame, and `prepare_position_data` raises
  instead of silently dropping features (this is what produced the old single-feature
  models fitted on price alone)
- split is by whole season — train 2016-17…2022-23, validation 2023-24,
  test 2024-25 + 2025-26
- scaler, correlation filter and PCA basis are fitted on the **training fold only**
- hyperparameter search uses `TimeSeriesSplit`, not shuffled `KFold`

Expect the test R² to come in **below** the numbers in the original notebook. The old
figures came from a shuffled split that let the model see neighbouring gameweeks; these
are honest season-holdout numbers.

In [ ]:
!python scripts/train.py

## 7. Results

In [ ]:
import json
import pandas as pd

metrics = json.load(open('model_metrics.json'))

rows = []
for approach in ('direct', 'pca'):
    for position, res in metrics.get(approach, {}).items():
        for name, m in res['models'].items():
            rows.append({
                'approach': approach,
                'position': position,
                'model': name,
                'train_r2': round(m['train_r2'], 4),
                'val_r2': round(m['val_r2'], 4),
                'test_r2': round(m['test_r2'], 4),
                'test_mae': round(m['test_mae'], 4),
            })

table = pd.DataFrame(rows).sort_values(['approach', 'position', 'test_r2'],
                                       ascending=[True, True, False])
display(table)

print("\nBest per position (by test R2):")
display(table.loc[table.groupby(['approach', 'position']).test_r2.idxmax()])

In [ ]:
# A large train/test gap is the thing to watch: it means the model is fitting
# season-specific noise rather than something that transfers.
table['gap'] = (table.train_r2 - table.test_r2).round(4)
display(table.sort_values('gap', ascending=False).head(10))

## 8. Take the results home

`saved_models/` holds the trained models, scalers, PCA transformers and feature lists.
If you used Drive, everything is already saved there and this cell is unnecessary.

In [ ]:
# !zip -qr saved_models.zip saved_models model_metrics.json
# from google.colab import files
# files.download('saved_models.zip')